# Cell 1 — config

In [1]:
from pathlib import Path
import os, random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "seed": SEED,
    "device": str(DEVICE),
    "thr": 0.5,
    "batch_size": 8,
    "num_workers": 2,
    "max_pr_pixels": 300_000,
}

print("DEVICE:", DEVICE)
print("CFG:", CFG)

DEVICE: cpu
CFG: {'seed': 42, 'device': 'cpu', 'thr': 0.5, 'batch_size': 8, 'num_workers': 2, 'max_pr_pixels': 300000}


# Cell 2 -Imports + dependency check

In [2]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + pkgs)

import cv2
import matplotlib.pyplot as plt

try:
    from sklearn.metrics import precision_recall_curve, auc
except Exception:
    pip_install(["scikit-learn"])
    from sklearn.metrics import precision_recall_curve, auc

try:
    import segmentation_models_pytorch as smp
except Exception:
    # Do NOT pin timm explicitly (it caused conflicts in your Kaggle env).
    pip_install(["segmentation-models-pytorch==0.3.3"])
    import segmentation_models_pytorch as smp

from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__)
print("cv2:", cv2.__version__)
print("smp:", smp.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 180.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 233.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 289.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 111.1 MB/s eta 0:00:00
torch: 2.9.0+cpu
cv2: 4.12.0
smp: 0.3.3


# Cell 3 — Locate inputs (auto-detect + explicit fallbacks)


In [3]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def find_dir_named(root: Path, name: str, must_contain=()):
    for p in root.rglob(name):
        if p.is_dir():
            ok = True
            for mc in must_contain:
                if not (p / mc).exists():
                    ok = False
                    break
            if ok:
                return p
    return None

KAGGLE_INPUT = Path("/kaggle/input")

# Dataset root (folder containing images/ and masks/)
SEG_ROOT = first_existing([
    "/kaggle/input/datasets/lakshamyiddha/crack-segmentation-dataset/crack_segmentation_dataset",
    "/kaggle/input/crack-segmentation-dataset/crack_segmentation_dataset",
])
if SEG_ROOT is None:
    SEG_ROOT = find_dir_named(KAGGLE_INPUT, "crack_segmentation_dataset", must_contain=("images", "masks"))

if SEG_ROOT is None:
    raise FileNotFoundError("Could not locate crack_segmentation_dataset with images/ and masks/ under /kaggle/input")

IMG_DIR = SEG_ROOT / "images"
MSK_DIR = SEG_ROOT / "masks"

VAL_SPLIT = first_existing([
    "/kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv",
    "/kaggle/input/phase-3-csv-files/val_split.csv",
])
if VAL_SPLIT is None:
    # search any val_split.csv
    hits = list(KAGGLE_INPUT.rglob("val_split.csv"))
    if len(hits) == 0:
        raise FileNotFoundError("Could not find val_split.csv under /kaggle/input")
    VAL_SPLIT = hits[0]

# Model checkpoints (explicit paths first; fall back to glob search)
CKPT_UNET = first_existing([
    "/kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt",
])
if CKPT_UNET is None:
    hits = list(KAGGLE_INPUT.rglob("unet_efficientnetb3_best_ep*.pt"))
    CKPT_UNET = hits[0] if hits else None

CKPT_DEEPLAB = first_existing([
    "/kaggle/input/datasets/arman00p/model-2-checkpointsdeeplabv3plus-efficientnet-b3/deeplabv3plus_efficientnetb3_pt files/deeplabv3plus_efficientnetb3_best_20260222_102303.pt",
])
if CKPT_DEEPLAB is None:
    hits = list(KAGGLE_INPUT.rglob("deeplabv3plus_efficientnetb3_best*.pt"))
    CKPT_DEEPLAB = hits[0] if hits else None

CKPT_UNETPP = first_existing([
    "/kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt",
])
if CKPT_UNETPP is None:
    hits = list(KAGGLE_INPUT.rglob("unetplusplus_efficientnetb3_best*.pt"))
    CKPT_UNETPP = hits[0] if hits else None

for name, p in [("SEG_ROOT", SEG_ROOT), ("VAL_SPLIT", VAL_SPLIT),
                ("CKPT_UNET", CKPT_UNET), ("CKPT_DEEPLAB", CKPT_DEEPLAB), ("CKPT_UNETPP", CKPT_UNETPP)]:
    print(f"{name}: {p} | exists:", (p is not None and Path(p).exists()))

if CKPT_UNET is None or CKPT_DEEPLAB is None or CKPT_UNETPP is None:
    raise FileNotFoundError("One or more checkpoint paths could not be located. Fix paths in Cell 3.")

SEG_ROOT: /kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset | exists: True
VAL_SPLIT: /kaggle/input/datasets/arman00p/phase-3-csv-files/val_split.csv | exists: True
CKPT_UNET: /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt | exists: True
CKPT_DEEPLAB: /kaggle/input/datasets/arman00p/model-2-checkpointsdeeplabv3plus-efficientnet-b3/deeplabv3plus_efficientnetb3_pt files/deeplabv3plus_efficientnetb3_best_20260222_102303.pt | exists: True
CKPT_UNETPP: /kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt | exists: True


# Cell 4 — SMP preprocessing (RGB + ImageNet encoder preprocessing)

In [4]:
try:
    from segmentation_models_pytorch.encoders import get_preprocessing_fn
except Exception:
    from segmentation_models_pytorch.encoders import get_preprocessing_fn  # re-import

def get_smp_preprocess_fn(encoder_name: str, pretrained: str = "imagenet"):
    # Support both signatures used across SMP versions
    try:
        return get_preprocessing_fn(encoder_name, pretrained=pretrained)
    except TypeError:
        return get_preprocessing_fn(encoder_name, encoder_weights=pretrained)

ENCODER_NAME = "efficientnet-b3"
ENCODER_PRETRAINED = "imagenet"
PREPROCESS_FN = get_smp_preprocess_fn(ENCODER_NAME, ENCODER_PRETRAINED)

print("Preprocess fn ready for:", ENCODER_NAME, ENCODER_PRETRAINED)

Preprocess fn ready for: efficientnet-b3 imagenet


# Cell 5 — Dataset + Loader (supports your val_split schema)

In [5]:
class CrackSegDataset(Dataset):
    def __init__(self, csv_path: Path, img_dir: Path, msk_dir: Path, size: int, preprocess_fn=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.msk_dir = Path(msk_dir)
        self.size = int(size)
        self.preprocess_fn = preprocess_fn

        cols = set(self.df.columns)

        # image path column
        if "image_path" in cols:
            self.img_col = "image_path"
        elif "image" in cols:
            self.img_col = "image"
        elif "filename" in cols:
            self.img_col = "filename"
        else:
            raise ValueError(f"Split CSV must contain image_path/image/filename. Found: {self.df.columns.tolist()}")

        # mask path column
        if "mask_path" in cols:
            self.msk_col = "mask_path"
        elif "mask" in cols:
            self.msk_col = "mask"
        else:
            raise ValueError(f"Split CSV must contain mask_path/mask. Found: {self.df.columns.tolist()}")

        # crack flag
        if "has_crack" in cols:
            self.has_crack_col = "has_crack"
        elif "empty_mask" in cols:
            self.has_crack_col = "empty_mask"  # will invert later
        else:
            self.has_crack_col = None

        # filename/stem for logging
        self.name_col = "stem" if "stem" in cols else None

    def _resolve_path(self, p: str, base_a: Path, base_b: Path):
        p = Path(str(p))
        if p.is_absolute() and p.exists():
            return p
        # try relative to SEG_ROOT
        if (SEG_ROOT / p).exists():
            return SEG_ROOT / p
        # try relative to image/mask dirs
        if (base_a / p).exists():
            return base_a / p
        if (base_b / p).exists():
            return base_b / p
        return p  # will fail later with clear error

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_p = self._resolve_path(row[self.img_col], self.img_dir, self.msk_dir)
        msk_p = self._resolve_path(row[self.msk_col], self.msk_dir, self.img_dir)

        img = cv2.imread(str(img_p), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_p}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)

        msk = cv2.imread(str(msk_p), cv2.IMREAD_GRAYSCALE)
        if msk is None:
            raise FileNotFoundError(f"Could not read mask: {msk_p}")
        msk = cv2.resize(msk, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        msk = (msk > 0).astype(np.float32)

        img = img.astype(np.float32) / 255.0
        if self.preprocess_fn is not None:
            img = self.preprocess_fn(img)

        img_t = torch.from_numpy(img.transpose(2, 0, 1)).float()
        msk_t = torch.from_numpy(msk[None, ...]).float()

        if self.has_crack_col is None:
            has_crack = float(msk_t.sum().item() > 0)
        else:
            v = float(row[self.has_crack_col])
            has_crack = (1.0 - v) if self.has_crack_col == "empty_mask" else v

        if self.name_col is not None:
            name = str(row[self.name_col])
        else:
            name = str(Path(img_p).name)

        return img_t, msk_t, name, float(has_crack)

def make_loader(csv_path: Path, size: int, batch_size: int, num_workers: int, preprocess_fn):
    ds = CrackSegDataset(csv_path, IMG_DIR, MSK_DIR, size=size, preprocess_fn=preprocess_fn)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        drop_last=False,
    )

# quick integrity check
dfv = pd.read_csv(VAL_SPLIT)
print("val_split rows:", len(dfv))

val_split rows: 1695


# Cell 6 — Robust checkpoint loader (handles model_state / model_state_dict + prefix stripping)

In [6]:
from collections import OrderedDict

def _is_state_dict_like(d):
    if not isinstance(d, dict):
        return False
    if len(d) == 0:
        return False
    # heuristic: most values are tensors
    tv = sum(isinstance(v, torch.Tensor) for v in d.values())
    return tv / max(1, len(d)) > 0.8

def _strip_prefixes(state_dict: dict):
    out = OrderedDict()
    for k, v in state_dict.items():
        nk = k
        for pref in ("module.", "model."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        out[nk] = v
    return out

def extract_state_dict_and_cfg(ckpt_obj):
    cfg = {}
    if isinstance(ckpt_obj, dict):
        # Common training formats
        for key in ("model_state_dict", "model_state", "state_dict", "model", "net"):
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                sd = ckpt_obj[key]
                cfg = ckpt_obj.get("config", ckpt_obj.get("cfg", {})) or {}
                return sd, cfg

        # If dict itself looks like a state_dict
        if _is_state_dict_like(ckpt_obj):
            return ckpt_obj, {}

    # If it's already an OrderedDict / state_dict-like
    if _is_state_dict_like(ckpt_obj):
        return ckpt_obj, {}

    raise ValueError("Unknown checkpoint format: cannot find model weights dict.")

def load_checkpoint(model: torch.nn.Module, ckpt_path: Path, min_overlap_ratio: float = 0.5):
    ckpt = torch.load(str(ckpt_path), map_location="cpu")
    sd_raw, cfg = extract_state_dict_and_cfg(ckpt)
    sd = _strip_prefixes(sd_raw)

    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(sd.keys())
    overlap = len(model_keys & ckpt_keys)
    ratio = overlap / max(1, len(model_keys))
    print(f"[load] overlap={overlap}/{len(model_keys)} ({ratio*100:.2f}%) | ckpt={Path(ckpt_path).name}")

    if ratio < min_overlap_ratio:
        raise RuntimeError(
            f"Checkpoint keys do not match model enough (overlap {ratio*100:.2f}%). "
            f"Likely wrong architecture or wrong key selection."
        )

    try:
        model.load_state_dict(sd, strict=True)
        strict_ok = True
        missing, unexpected = [], []
    except Exception:
        strict_ok = False
        missing, unexpected = model.load_state_dict(sd, strict=False)
        print(f"[WARN] strict=False load | missing={len(missing)} unexpected={len(unexpected)}")

    return cfg, strict_ok

# Cell 7 — Model builders (Model-1 / Model-2 / Model-3)

In [7]:
def build_unet():
    return smp.Unet(
        encoder_name="efficientnet-b3",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None,
    )

def build_deeplab():
    return smp.DeepLabV3Plus(
        encoder_name="efficientnet-b3",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None,
    )

def build_unetpp():
    return smp.UnetPlusPlus(
        encoder_name="efficientnet-b3",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
        activation=None,
    )

print("Builders ready.")

Builders ready.


# Cell 8 — Metrics helpers (IoU/Dice + safe PR sampling)

In [8]:
def iou_from_binary(pred: torch.Tensor, gt: torch.Tensor, eps: float = 1e-7):
    # pred, gt: [B,1,H,W] in {0,1}
    inter = (pred * gt).sum(dim=(1,2,3))
    union = (pred + gt - pred*gt).sum(dim=(1,2,3))
    iou = torch.where(union > 0, inter / (union + eps), torch.ones_like(union))
    return iou

def dice_from_binary(pred: torch.Tensor, gt: torch.Tensor, eps: float = 1e-7):
    inter = (pred * gt).sum(dim=(1,2,3))
    s = pred.sum(dim=(1,2,3)) + gt.sum(dim=(1,2,3))
    dice = torch.where(s > 0, (2*inter) / (s + eps), torch.ones_like(s))
    return dice

# Cell 9 — Standardized evaluator (fixes BFloat16 → NumPy)

In [9]:
@torch.no_grad()
def eval_model(model_name: str, builder, ckpt_path: Path, img_size: int, batch_size: int):
    model = builder()
    cfg, strict_ok = load_checkpoint(model, ckpt_path, min_overlap_ratio=0.5)
    model = model.to(DEVICE).eval()

    loader = make_loader(
        VAL_SPLIT,
        size=img_size,
        batch_size=batch_size,
        num_workers=CFG["num_workers"],
        preprocess_fn=PREPROCESS_FN,
    )

    thr = float(CFG["thr"])
    max_pr = int(CFG["max_pr_pixels"])

    iou_all = []
    iou_crack = []
    dice_crack = []

    no_crack_fp = 0
    no_crack_total = 0

    y_true_buf = []
    y_prob_buf = []
    sampled = 0

    for x, y, fn, has_crack in loader:
        x = x.to(DEVICE, dtype=torch.float32, non_blocking=True)
        y = y.to(DEVICE, dtype=torch.float32, non_blocking=True)

        logits = model(x)
        probs = torch.sigmoid(logits).float()  # critical: avoid bfloat16 numpy conversion 
        pred = (probs > thr).float()

        iou_b = iou_from_binary(pred, y).detach().cpu().numpy()
        iou_all.extend(iou_b.tolist())

        # crack-only mask (from GT)
        gt_has = (y.sum(dim=(1,2,3)) > 0).detach().cpu().numpy().astype(bool)
        if gt_has.any():
            iou_crack.extend(iou_b[gt_has].tolist())
            dice_b = dice_from_binary(pred, y).detach().cpu().numpy()
            dice_crack.extend(dice_b[gt_has].tolist())

        # no-crack false positive (any predicted pixel)
        pred_any = (pred.sum(dim=(1,2,3)) > 0).detach().cpu().numpy().astype(bool)
        for g, p in zip(gt_has, pred_any):
            if not g:
                no_crack_total += 1
                if p:
                    no_crack_fp += 1

        # PR-AUC sampling (cap total pixels)
        if sampled < max_pr:
            flat_y = y.detach().flatten().cpu().numpy().astype(np.uint8)
            flat_p = probs.detach().flatten().cpu().numpy().astype(np.float32)

            remaining = max_pr - sampled
            take = min(remaining, flat_y.size)
            if take > 0:
                y_true_buf.append(flat_y[:take])
                y_prob_buf.append(flat_p[:take])
                sampled += take

    y_true = np.concatenate(y_true_buf) if len(y_true_buf) else np.array([], dtype=np.uint8)
    y_prob = np.concatenate(y_prob_buf) if len(y_prob_buf) else np.array([], dtype=np.float32)

    if y_true.size > 0:
        prec, rec, _ = precision_recall_curve(y_true, y_prob)
        pr_auc = float(auc(rec, prec))
    else:
        pr_auc = float("nan")

    out = {
        "MODEL_NAME": model_name,
        "IMG_SIZE": int(img_size),
        "mean_iou_all": float(np.mean(iou_all)) if len(iou_all) else float("nan"),
        "median_iou_all": float(np.median(iou_all)) if len(iou_all) else float("nan"),
        "mean_iou_crack_only": float(np.mean(iou_crack)) if len(iou_crack) else float("nan"),
        "mean_dice_crack_only": float(np.mean(dice_crack)) if len(dice_crack) else float("nan"),
        "pixel_pr_auc": pr_auc,
        "no_crack_false_positive_rate": float(no_crack_fp / max(1, no_crack_total)),
        "no_crack_false_positive_count": int(no_crack_fp),
        "ckpt": str(ckpt_path),
        "strict_ok": bool(strict_ok),
    }
    return out

# Cell 10 — Run Model-1/2/3 standardized evaluation (native sizes)

In [10]:
import sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"] + pkgs)

try:
    import segmentation_models_pytorch as smp
except Exception:
    pip_install(["segmentation-models-pytorch"])
    import segmentation_models_pytorch as smp

try:
    from sklearn.metrics import precision_recall_curve, auc
except Exception:
    pip_install(["scikit-learn"])
    from sklearn.metrics import precision_recall_curve, auc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _extract_state_dict(obj):
    if isinstance(obj, dict):
        for k in ["model_state_dict", "model_state", "state_dict", "model"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k], obj.get("config", {}) or obj.get("cfg", {}) or {}
        if all(isinstance(v, torch.Tensor) for v in obj.values()):
            return obj, {}
    raise ValueError("Could not find a state_dict inside checkpoint.")

def _strip_prefix(sd):
    sd = dict(sd)
    prefixes = ["module.", "model.", "net."]
    for p in prefixes:
        ks = list(sd.keys())
        if len(ks) > 0 and sum(k.startswith(p) for k in ks[: min(50, len(ks))]) > 0.8 * min(50, len(ks)):
            sd = {k[len(p):]: v for k, v in sd.items()}
    return sd

def _guess_unet_encoder(sd_keys):
    sd_keys = list(sd_keys)
    if any(k.startswith("encoder._conv_stem") for k in sd_keys) or any("encoder._blocks" in k for k in sd_keys):
        return "efficientnet-b3"
    if any(k.startswith("encoder.conv_stem") for k in sd_keys) or any("encoder.blocks" in k for k in sd_keys):
        return "timm-efficientnet-b3"
    if any(k.startswith("encoder.model.conv_stem") for k in sd_keys) or any("encoder.model.blocks" in k for k in sd_keys):
        return "timm-efficientnet-b3"
    return "timm-efficientnet-b3"

def _prep_params(encoder_name):
    params = smp.encoders.get_preprocessing_params(encoder_name, pretrained="imagenet")
    mean = np.array(params["mean"], dtype=np.float32)
    std = np.array(params["std"], dtype=np.float32)
    input_range = params.get("input_range", [0, 1])
    return mean, std, input_range

def _preprocess_rgb(img_rgb, mean, std, input_range):
    x = img_rgb.astype(np.float32)
    if float(input_range[1]) <= 1.0:
        x = x / 255.0
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))
    return x

class CrackSegDataset(Dataset):
    def __init__(self, csv_path: Path, size: int, mean, std, input_range):
        self.df = pd.read_csv(csv_path)
        if "image_path" in self.df.columns:
            self.im_col = "image_path"
        elif "image" in self.df.columns:
            self.im_col = "image"
        elif "filename" in self.df.columns:
            self.im_col = "filename"
        else:
            raise ValueError(f"CSV missing image column. Found: {self.df.columns.tolist()}")

        if "mask_path" in self.df.columns:
            self.mk_col = "mask_path"
        elif "mask" in self.df.columns:
            self.mk_col = "mask"
        else:
            raise ValueError(f"CSV missing mask column. Found: {self.df.columns.tolist()}")

        self.name_col = "stem" if "stem" in self.df.columns else self.im_col
        self.size = int(size)
        self.mean, self.std, self.input_range = mean, std, input_range

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        ip = str(r[self.im_col])
        mp = str(r[self.mk_col])

        img = cv2.imread(ip, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"Image not found: {ip}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        msk = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if msk is None:
            raise FileNotFoundError(f"Mask not found: {mp}")
        msk = (msk > 127).astype(np.float32)

        if self.size is not None:
            img = cv2.resize(img, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk, (self.size, self.size), interpolation=cv2.INTER_NEAREST)

        x = _preprocess_rgb(img, self.mean, self.std, self.input_range)
        x = torch.from_numpy(x).float()
        y = torch.from_numpy(msk[None, ...]).float()

        name = str(r[self.name_col])
        return x, y, name

def _iou_dice_from_probs(probs, y, thr=0.5, eps=1e-7):
    p = (probs > thr)
    t = (y > 0.5)

    inter = (p & t).sum().item()
    union = (p | t).sum().item()
    iou = inter / (union + eps)

    p_sum = p.sum().item()
    t_sum = t.sum().item()
    dice = (2 * inter) / (p_sum + t_sum + eps)
    return float(iou), float(dice)

def _load_model(model, ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd, cfg = _extract_state_dict(ckpt)
    sd = _strip_prefix(sd)

    model_sd = model.state_dict()
    overlap = len(set(model_sd.keys()) & set(sd.keys()))
    ratio = overlap / max(1, len(model_sd))
    print(f"[load] overlap={overlap}/{len(model_sd)} ({ratio*100:.2f}%) | ckpt={ckpt_path.name}")

    if ratio < 0.50:
        raise RuntimeError(
            f"Checkpoint keys do not match model enough (overlap {ratio*100:.2f}%). "
            f"Most common cause: wrong encoder_name (timm vs efficientnet)."
        )

    res = model.load_state_dict(sd, strict=False)
    missing = len(getattr(res, "missing_keys", []))
    unexpected = len(getattr(res, "unexpected_keys", []))
    if missing or unexpected:
        print(f"[load] strict=False | missing={missing} unexpected={unexpected}")
    else:
        print("[load] strict OK")
    return cfg

def build_unet_from_ckpt(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd, _ = _extract_state_dict(ckpt)
    sd = _strip_prefix(sd)
    enc = _guess_unet_encoder(sd.keys())

    if enc not in smp.encoders.get_encoder_names():
        raise RuntimeError(f"Encoder '{enc}' not available in this smp install.")

    model = smp.Unet(
        encoder_name=enc,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model, enc

def build_deeplab():
    enc = "efficientnet-b3"
    model = smp.DeepLabV3Plus(
        encoder_name=enc,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model, enc

def build_unetpp():
    enc = "efficientnet-b3"
    model = smp.UnetPlusPlus(
        encoder_name=enc,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None,
    )
    return model, enc

def eval_model(model_name: str, model_enc_builder, ckpt_path: Path, img_size: int, batch_size: int = 8, max_pr_pixels: int = 300_000):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(str(ckpt_path))

    model, enc = model_enc_builder(ckpt_path) if model_enc_builder is build_unet_from_ckpt else model_enc_builder()
    cfg = _load_model(model, ckpt_path)

    mean, std, input_range = _prep_params(enc)

    vs = Path(VAL_SPLIT) if "VAL_SPLIT" in globals() else Path(val_split_csv)
    ds = CrackSegDataset(vs, size=img_size, mean=mean, std=std, input_range=input_range)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=(DEVICE.type == "cuda"))

    model = model.to(DEVICE).eval()

    iou_all, dice_all = [], []
    iou_cr, dice_cr = [], []
    no_crack_total = 0
    no_crack_fp = 0

    sampled = 0
    y_flat = []
    p_flat = []

    with torch.inference_mode():
        pbar = tqdm(loader, desc=f"eval {model_name}", leave=False)
        for x, y, _ in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            use_amp = (DEVICE.type == "cuda")
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp, dtype=torch.float16 if use_amp else torch.float32):
                logits = model(x)
                probs = torch.sigmoid(logits)

            probs = probs.float()

            bs = x.size(0)
            for i in range(bs):
                yi = y[i:i+1]
                pi = probs[i:i+1]

                iou_i, dice_i = _iou_dice_from_probs(pi, yi, thr=0.5)
                iou_all.append(iou_i)
                dice_all.append(dice_i)

                has_crack = (yi > 0.5).any().item()
                if has_crack:
                    iou_cr.append(iou_i)
                    dice_cr.append(dice_i)
                else:
                    no_crack_total += 1
                    if (pi > 0.5).any().item():
                        no_crack_fp += 1

            if sampled < max_pr_pixels:
                flat_y = y.detach().flatten().to("cpu").numpy().astype(np.uint8)
                flat_p = probs.detach().flatten().float().to("cpu").numpy().astype(np.float32)
                remaining = max_pr_pixels - sampled
                take = min(remaining, flat_y.size)
                if take > 0:
                    idx = np.random.choice(flat_y.size, size=take, replace=False)
                    y_flat.append(flat_y[idx])
                    p_flat.append(flat_p[idx])
                    sampled += take

    pixel_pr_auc = np.nan
    if sampled > 0:
        y_all = np.concatenate(y_flat)
        p_all = np.concatenate(p_flat)
        prec, rec, _ = precision_recall_curve(y_all, p_all)
        pixel_pr_auc = float(auc(rec, prec))

    out = {
        "MODEL_NAME": model_name,
        "IMG_SIZE": int(img_size),
        "mean_iou_all": float(np.mean(iou_all)) if len(iou_all) else np.nan,
        "mean_iou_crack_only": float(np.mean(iou_cr)) if len(iou_cr) else np.nan,
        "mean_dice_crack_only": float(np.mean(dice_cr)) if len(dice_cr) else np.nan,
        "pixel_pr_auc": pixel_pr_auc,
        "no_crack_false_positive_rate": float(no_crack_fp / max(1, no_crack_total)),
        "no_crack_false_positive_count": int(no_crack_fp),
        "ckpt": str(ckpt_path),
    }
    return out

summaries = []
summaries.append(eval_model("unet_efficientnetb3", build_unet_from_ckpt, CKPT_UNET, img_size=256, batch_size=8))
summaries.append(eval_model("deeplabv3plus_efficientnetb3", build_deeplab, CKPT_DEEPLAB, img_size=448, batch_size=8))
summaries.append(eval_model("unetplusplus_efficientnetb3", build_unetpp, CKPT_UNETPP, img_size=448, batch_size=8))

cmp_raw = pd.DataFrame(summaries)
cmp_raw

[load] overlap=634/634 (100.00%) | ckpt=unet_efficientnetb3_best_ep029_20260219_205815.pt
[load] strict OK


eval unet_efficientnetb3:   0%|          | 0/212 [00:00<?, ?it/s]

[load] overlap=633/633 (100.00%) | ckpt=deeplabv3plus_efficientnetb3_best_20260222_102303.pt
[load] strict OK


eval deeplabv3plus_efficientnetb3:   0%|          | 0/212 [00:00<?, ?it/s]

[load] overlap=706/706 (100.00%) | ckpt=unetplusplus_efficientnetb3_best.pt
[load] strict OK


eval unetplusplus_efficientnetb3:   0%|          | 0/212 [00:00<?, ?it/s]

,MODEL_NAME,IMG_SIZE,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,pixel_pr_auc,no_crack_false_positive_rate,no_crack_false_positive_count,ckpt
0,unet_efficientnetb3,256,0.373325,0.428427,0.577456,0.715222,0.009174,2,/kaggle/input/datasets/arman00p/model-1-checkp...
1,deeplabv3plus_efficientnetb3,448,0.499137,0.572808,0.713919,0.802646,0.004587,1,/kaggle/input/datasets/arman00p/model-2-checkp...
2,unetplusplus_efficientnetb3,448,0.506005,0.580690,0.721545,0.808927,0.027523,6,/kaggle/input/datasets/arman00p/model-3-checkp...


# Cell 12 — Optional: Model-1 sensitivity to inference resolution (256 vs 448)

In [11]:
from pathlib import Path
import pandas as pd

# Always ensure output directories exist (even if RUN_DIR is already defined)
OUT = Path(globals().get("OUT", "/kaggle/working/output"))
OUT.mkdir(parents=True, exist_ok=True)

RUN_NAME = globals().get("RUN_NAME", "phase5_compare")
RUN_DIR = Path(globals().get("RUN_DIR", OUT / "phase5" / RUN_NAME))
FIG_DIR = Path(globals().get("FIG_DIR", RUN_DIR / "figures"))

RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Builder: prefer ckpt-aware builder if present
try:
    builder = build_unet_from_ckpt
except NameError:
    def builder(*args, **kwargs):
        return build_unet(), "efficientnet-b3"

# Robust batch-size fetch
bs = 8
if isinstance(globals().get("CFG", None), dict):
    bs = int(CFG.get("batch_size", 8))
else:
    bs = int(getattr(globals().get("CFG", None), "batch_size", 8))

sens = []
sens.append(eval_model("unet_efficientnetb3_infer256", builder, CKPT_UNET, img_size=256, batch_size=bs))
sens.append(eval_model("unet_efficientnetb3_infer448", builder, CKPT_UNET, img_size=448, batch_size=bs))

sens_df = pd.DataFrame(sens)
sens_csv = RUN_DIR / "phase5_model1_resolution_sensitivity.csv"
sens_df.to_csv(sens_csv, index=False)

print("Saved:", sens_csv)
sens_df

[load] overlap=634/634 (100.00%) | ckpt=unet_efficientnetb3_best_ep029_20260219_205815.pt
[load] strict OK


eval unet_efficientnetb3_infer256:   0%|          | 0/212 [00:00<?, ?it/s]

[load] overlap=634/634 (100.00%) | ckpt=unet_efficientnetb3_best_ep029_20260219_205815.pt
[load] strict OK


eval unet_efficientnetb3_infer448:   0%|          | 0/212 [00:00<?, ?it/s]

Saved: /kaggle/working/output/phase5/phase5_compare/phase5_model1_resolution_sensitivity.csv


,MODEL_NAME,IMG_SIZE,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,pixel_pr_auc,no_crack_false_positive_rate,no_crack_false_positive_count,ckpt
0,unet_efficientnetb3_infer256,256,0.373325,0.428427,0.577456,0.725546,0.009174,2,/kaggle/input/datasets/arman00p/model-1-checkp...
1,unet_efficientnetb3_infer448,448,0.481037,0.552036,0.695101,0.801882,0.036697,8,/kaggle/input/datasets/arman00p/model-1-checkp...


# Cell 13 — Reload comparison results (robust) + regenerate a clean table PNG/CSV/TEX

In [12]:
from pathlib import Path
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reuse existing RUN_DIR/FIG_DIR if your notebook already set them
RUN_DIR = globals().get("RUN_DIR", Path("/kaggle/working/output/phase5/phase5_compare"))
FIG_DIR = globals().get("FIG_DIR", RUN_DIR / "figures")
RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def _pick_base_df():
    if isinstance(globals().get("cmp_df", None), pd.DataFrame) and len(cmp_df) > 0:
        return cmp_df.copy()
    if isinstance(globals().get("cmp_raw", None), pd.DataFrame) and len(cmp_raw) > 0:
        return cmp_raw.copy()
    if isinstance(globals().get("summaries", None), list) and len(summaries) > 0:
        return pd.DataFrame(summaries).copy()

    # fallback: find any saved comparison table
    candidates = []
    for root in [RUN_DIR, FIG_DIR, Path("/kaggle/working/output"), Path("/kaggle/working")]:
        if root.exists():
            candidates += list(root.rglob("phase5_comparison_table*.csv"))
    if candidates:
        return pd.read_csv(sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0])

    raise FileNotFoundError("No in-memory results found (cmp_df/cmp_raw/summaries) and no saved phase5_comparison_table*.csv found.")

base = _pick_base_df()

# Normalize column names (support older keys)
rename_map = {
    "model": "MODEL_NAME",
    "img_size": "IMG_SIZE",
    "val_iou_mean": "mean_iou_all",
    "val_iou_median": "median_iou_all",
    "val_acc_mean": "pixel_acc_mean",
    "pixel_acc_mean": "pixel_acc_mean",
    "pixel_pr_auc": "pixel_pr_auc",
}
base = base.rename(columns={k: v for k, v in rename_map.items() if k in base.columns})

need_cols = [
    "MODEL_NAME","IMG_SIZE",
    "mean_iou_all","mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc",
    "no_crack_false_positive_rate","no_crack_false_positive_count","ckpt"
]
for c in need_cols:
    if c not in base.columns:
        base[c] = np.nan

cmp_df = base[need_cols].copy()

# Types
cmp_df["MODEL_NAME"] = cmp_df["MODEL_NAME"].astype(str)
cmp_df["IMG_SIZE"] = pd.to_numeric(cmp_df["IMG_SIZE"], errors="coerce")
for c in ["mean_iou_all","mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc",
          "no_crack_false_positive_rate","no_crack_false_positive_count"]:
    cmp_df[c] = pd.to_numeric(cmp_df[c], errors="coerce")

# Shorten checkpoint path for paper table
def _short_ckpt(s: str, keep=42):
    s = str(s) if s is not None else ""
    name = Path(s).name if s.strip() else s
    if len(name) <= keep:
        return name
    return name[: (keep//2 - 1)] + "…" + name[-(keep//2 - 1):]

cmp_df["ckpt"] = cmp_df["ckpt"].astype(str).map(_short_ckpt)

# Sort (primary: crack-only IoU, secondary: PR-AUC)
cmp_df = cmp_df.sort_values(["mean_iou_crack_only","pixel_pr_auc"], ascending=[False, False], na_position="last").reset_index(drop=True)

# Save machine-readable artifacts
csv_out = RUN_DIR / "phase5_comparison_table_paper_ready.csv"
md_out  = RUN_DIR / "phase5_comparison_table_paper_ready.md"
tex_out = RUN_DIR / "phase5_comparison_table_paper_ready.tex"
cmp_df.to_csv(csv_out, index=False)
try:
    md_out.write_text(cmp_df.to_markdown(index=False))
except Exception:
    md_out.write_text(cmp_df.to_string(index=False))
tex_out.write_text(cmp_df.to_latex(index=False, float_format="%.6f"))

# Render a clean PNG (no overlap)
show = cmp_df.copy()
fmt_float = ["mean_iou_all","mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc","no_crack_false_positive_rate"]
for c in fmt_float:
    show[c] = show[c].map(lambda v: "" if pd.isna(v) else f"{v:.6f}")
show["no_crack_false_positive_count"] = show["no_crack_false_positive_count"].map(lambda v: "" if pd.isna(v) else f"{int(v)}")
show["IMG_SIZE"] = show["IMG_SIZE"].map(lambda v: "" if pd.isna(v) else f"{int(v)}")
show["ckpt"] = show["ckpt"].map(lambda s: "\n".join(textwrap.wrap(str(s), width=28)))

# winners (bold best / lowest FP)
best_high = ["mean_iou_all","mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc"]
best_low  = ["no_crack_false_positive_rate","no_crack_false_positive_count"]
winners = {}
for c in best_high:
    winners[c] = int(pd.to_numeric(cmp_df[c], errors="coerce").idxmax())
for c in best_low:
    winners[c] = int(pd.to_numeric(cmp_df[c], errors="coerce").idxmin())

plt.figure(figsize=(20, 2.2 + 0.75*len(show)))
plt.axis("off")

col_widths = [0.16, 0.06, 0.08, 0.10, 0.10, 0.08, 0.12, 0.10, 0.20]
col_widths = col_widths[:len(show.columns)]  # safety

tbl = plt.table(
    cellText=show.astype(str).values.tolist(),
    colLabels=show.columns.tolist(),
    cellLoc="center",
    loc="center",
    colWidths=col_widths,
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)

col_idx = {n:i for i,n in enumerate(show.columns)}
for metric, ridx in winners.items():
    tbl[(ridx+1, col_idx[metric])].get_text().set_fontweight("bold")

png_out = FIG_DIR / "phase5_comparison_table_paper_ready.png"
plt.savefig(png_out, dpi=300, bbox_inches="tight")
plt.close()

print("Saved:", csv_out)
print("Saved:", md_out)
print("Saved:", tex_out)
print("Saved:", png_out)

cmp_df

Saved: /kaggle/working/output/phase5/phase5_compare/phase5_comparison_table_paper_ready.csv
Saved: /kaggle/working/output/phase5/phase5_compare/phase5_comparison_table_paper_ready.md
Saved: /kaggle/working/output/phase5/phase5_compare/phase5_comparison_table_paper_ready.tex
Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_comparison_table_paper_ready.png


,MODEL_NAME,IMG_SIZE,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,pixel_pr_auc,no_crack_false_positive_rate,no_crack_false_positive_count,ckpt
0,unetplusplus_efficientnetb3,448,0.506005,0.580690,0.721545,0.808927,0.027523,6,unetplusplus_efficientnetb3_best.pt
1,deeplabv3plus_efficientnetb3,448,0.499137,0.572808,0.713919,0.802646,0.004587,1,deeplabv3plus_effici…t_20260222_102303.pt
2,unet_efficientnetb3,256,0.373325,0.428427,0.577456,0.715222,0.009174,2,unet_efficientnetb3_…9_20260219_205815.pt


# Cell 14 — Extra paper figures (bars + tradeoff + Model-1 resolution plot)

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUN_DIR = globals().get("RUN_DIR", Path("/kaggle/working/output/phase5/phase5_compare"))
FIG_DIR = globals().get("FIG_DIR", RUN_DIR / "figures")
RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

cmp_csv = RUN_DIR / "phase5_comparison_table_paper_ready.csv"
cmp_df = pd.read_csv(cmp_csv)

x = cmp_df["MODEL_NAME"].astype(str).tolist()

metrics_hi = ["mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc"]
vals = np.vstack([pd.to_numeric(cmp_df[m], errors="coerce").values for m in metrics_hi])

plt.figure(figsize=(10, 4.8))
idx = np.arange(len(x))
w = 0.26
for i, m in enumerate(metrics_hi):
    plt.bar(idx + (i-1)*w, vals[i], width=w, label=m)
plt.xticks(idx, x, rotation=15, ha="right")
plt.ylabel("Score")
plt.title("Phase-5: Key crack-focused metrics (higher is better)")
plt.legend()
fig1 = FIG_DIR / "phase5_metrics_bar.png"
plt.tight_layout()
plt.savefig(fig1, dpi=300)
plt.close()

plt.figure(figsize=(8.5, 4.6))
fp = pd.to_numeric(cmp_df["no_crack_false_positive_rate"], errors="coerce").values
plt.bar(np.arange(len(x)), fp)
plt.xticks(np.arange(len(x)), x, rotation=15, ha="right")
plt.ylabel("False Positive Rate")
plt.title("Phase-5: No-crack false positive rate (lower is better)")
fig2 = FIG_DIR / "phase5_fp_rate_bar.png"
plt.tight_layout()
plt.savefig(fig2, dpi=300)
plt.close()

plt.figure(figsize=(7.2, 5.2))
iou_c = pd.to_numeric(cmp_df["mean_iou_crack_only"], errors="coerce").values
plt.scatter(fp, iou_c)
for i, name in enumerate(x):
    plt.annotate(name, (fp[i], iou_c[i]), textcoords="offset points", xytext=(6, 6))
plt.xlabel("No-crack false positive rate (lower better)")
plt.ylabel("Mean IoU (crack-only) (higher better)")
plt.title("Phase-5 trade-off: accuracy vs false alarms")
fig3 = FIG_DIR / "phase5_tradeoff_scatter.png"
plt.tight_layout()
plt.savefig(fig3, dpi=300)
plt.close()

def _find_one(filename: str, roots):
    roots = [Path(r) for r in roots if r is not None]
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.rglob(filename))
            if len(hits) > 0:
                return hits[0]
    raise FileNotFoundError(f"Could not find {filename}")

sens_csv = _find_one(
    "phase5_model1_resolution_sensitivity.csv",
    [RUN_DIR, FIG_DIR, Path("/kaggle/working"), Path("/kaggle/working/output")]
)
sens = pd.read_csv(sens_csv)
sens["IMG_SIZE"] = pd.to_numeric(sens["IMG_SIZE"], errors="coerce")
sens["mean_iou_crack_only"] = pd.to_numeric(sens["mean_iou_crack_only"], errors="coerce")

sens = sens.sort_values("IMG_SIZE")

plt.figure(figsize=(7.8, 4.6))
plt.plot(sens["IMG_SIZE"].values, sens["mean_iou_crack_only"].values, marker="o")
plt.xticks(sens["IMG_SIZE"].values)
plt.xlabel("Inference IMG_SIZE")
plt.ylabel("Mean IoU (crack-only)")
plt.title("Model-1 resolution sensitivity (crack-only IoU)")
fig4 = FIG_DIR / "phase5_model1_resolution_sensitivity_plot.png"
plt.tight_layout()
plt.savefig(fig4, dpi=300)
plt.close()

print("Saved:", fig1)
print("Saved:", fig2)
print("Saved:", fig3)
print("Saved:", fig4)

Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_metrics_bar.png
Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_fp_rate_bar.png
Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_tradeoff_scatter.png
Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_model1_resolution_sensitivity_plot.png


In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure output dirs exist
if "RUN_DIR" not in globals():
    OUT = globals().get("OUT", Path("/kaggle/working/output"))
    RUN_NAME = globals().get("RUN_NAME", "phase5_compare")
    RUN_DIR = Path(OUT) / "phase5" / RUN_NAME
    FIG_DIR = RUN_DIR / "figures"
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)

# Get comparison dataframe robustly
cmp_candidates = []
if "cmp_df" in globals() and isinstance(cmp_df, pd.DataFrame):
    cmp_candidates.append(cmp_df.copy())
if "cmp_raw" in globals() and isinstance(cmp_raw, pd.DataFrame):
    cmp_candidates.append(cmp_raw.copy())
if "summaries" in globals() and isinstance(summaries, list) and len(summaries) > 0:
    cmp_candidates.append(pd.DataFrame(summaries).copy())

if len(cmp_candidates) > 0:
    cmp_df_local = cmp_candidates[-1].copy()
else:
    cmp_path = RUN_DIR / "phase5_comparison_table.csv"
    if not cmp_path.exists():
        raise FileNotFoundError("No cmp_df/cmp_raw/summaries in memory and no phase5_comparison_table.csv found in RUN_DIR.")
    cmp_df_local = pd.read_csv(cmp_path)

# Normalize expected columns (in case older names exist)
rename_map = {
    "model": "MODEL_NAME",
    "img_size": "IMG_SIZE",
    "val_iou_mean": "mean_iou_all",
    "val_iou_crack_mean": "mean_iou_crack_only",
    "val_dice_crack_mean": "mean_dice_crack_only",
    "pixel_pr_auc": "pixel_pr_auc",
    "no_crack_false_positive_rate": "no_crack_false_positive_rate",
    "no_crack_false_positive_count": "no_crack_false_positive_count",
}
cmp_df_local = cmp_df_local.rename(columns={k: v for k, v in rename_map.items() if k in cmp_df_local.columns})

need_cols = [
    "MODEL_NAME","IMG_SIZE",
    "mean_iou_all","mean_iou_crack_only","mean_dice_crack_only","pixel_pr_auc",
    "no_crack_false_positive_rate","no_crack_false_positive_count","ckpt"
]
for c in need_cols:
    if c not in cmp_df_local.columns:
        cmp_df_local[c] = np.nan

paper_df = cmp_df_local[need_cols].copy()
paper_df["ckpt"] = paper_df["ckpt"].apply(lambda s: Path(str(s)).name if pd.notna(s) else "")

# Sort for display
paper_df = paper_df.sort_values("mean_iou_crack_only", ascending=False, na_position="last").reset_index(drop=True)

hi_cols_high = ["mean_iou_all", "mean_iou_crack_only", "mean_dice_crack_only", "pixel_pr_auc"]
hi_cols_low  = ["no_crack_false_positive_rate", "no_crack_false_positive_count"]

disp = paper_df.copy()
for c in hi_cols_high + hi_cols_low + ["IMG_SIZE"]:
    disp[c] = pd.to_numeric(disp[c], errors="coerce")

winners = {}
for c in hi_cols_high:
    winners[c] = int(np.nanargmax(disp[c].values)) if np.isfinite(disp[c].values).any() else None
for c in hi_cols_low:
    winners[c] = int(np.nanargmin(disp[c].values)) if np.isfinite(disp[c].values).any() else None

show = paper_df.copy()
for c in hi_cols_high + hi_cols_low + ["IMG_SIZE"]:
    show[c] = pd.to_numeric(show[c], errors="coerce")
for c in hi_cols_high + hi_cols_low:
    show[c] = show[c].round(6)

paper_csv = RUN_DIR / "phase5_comparison_table_paper_ready.csv"
paper_png = FIG_DIR / "phase5_comparison_table_paper_ready.png"
show.to_csv(paper_csv, index=False)

# Build a readable table PNG (no overlap)
cell_text = []
for _, r in show.iterrows():
    row = []
    for col in show.columns:
        v = r[col]
        if col in ("MODEL_NAME", "ckpt"):
            row.append(str(v))
        else:
            row.append("" if pd.isna(v) else f"{float(v):.6f}".rstrip("0").rstrip("."))
    cell_text.append(row)

col_lens = []
for j, col in enumerate(show.columns):
    mx = max([len(str(col))] + [len(str(cell_text[i][j])) for i in range(len(cell_text))])
    col_lens.append(mx)
col_lens = np.array(col_lens, dtype=float)
col_widths = (col_lens / col_lens.sum()).tolist()

plt.figure(figsize=(18, 2.4 + 0.55*len(show)))
plt.axis("off")
tbl = plt.table(
    cellText=cell_text,
    colLabels=show.columns.tolist(),
    cellLoc="center",
    loc="center",
    colWidths=col_widths
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.3)

# Bold winner cells
col_idx = {name:i for i,name in enumerate(show.columns)}
for metric, ridx in winners.items():
    if ridx is None:
        continue
    r = ridx + 1  # header row is 0
    c = col_idx[metric]
    tbl[(r, c)].get_text().set_fontweight("bold")

plt.savefig(paper_png, dpi=300, bbox_inches="tight")
plt.close()

print("Saved:", paper_csv)
print("Saved:", paper_png)
show

Saved: /kaggle/working/output/phase5/phase5_compare/phase5_comparison_table_paper_ready.csv
Saved: /kaggle/working/output/phase5/phase5_compare/figures/phase5_comparison_table_paper_ready.png


,MODEL_NAME,IMG_SIZE,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,pixel_pr_auc,no_crack_false_positive_rate,no_crack_false_positive_count,ckpt
0,unetplusplus_efficientnetb3,448,0.506005,0.580690,0.721545,0.808927,0.027523,6,unetplusplus_efficientnetb3_best.pt
1,deeplabv3plus_efficientnetb3,448,0.499137,0.572808,0.713919,0.802646,0.004587,1,deeplabv3plus_efficientnetb3_best_20260222_102...
2,unet_efficientnetb3,256,0.373325,0.428427,0.577456,0.715222,0.009174,2,unet_efficientnetb3_best_ep029_20260219_205815.pt


# Cell 15 — Final deliverable zip (small, only Phase-5 outputs)

In [15]:
import shutil, time
from pathlib import Path

RUN_DIR = globals().get("RUN_DIR", Path("/kaggle/working/output/phase5/phase5_compare"))
FIG_DIR = globals().get("FIG_DIR", RUN_DIR / "figures")
RUN_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

stamp = time.strftime("%Y%m%d_%H%M%S")
deliver_dir = RUN_DIR / f"deliverable_{stamp}"
deliver_dir.mkdir(parents=True, exist_ok=True)

def _maybe_copy(p: Path):
    if p is not None and Path(p).exists():
        shutil.copy2(str(p), str(deliver_dir / Path(p).name))

# Core Phase-5 artifacts
_maybe_copy(RUN_DIR / "phase5_comparison_table_paper_ready.csv")
_maybe_copy(RUN_DIR / "phase5_comparison_table_paper_ready.md")
_maybe_copy(RUN_DIR / "phase5_comparison_table_paper_ready.tex")
_maybe_copy(FIG_DIR / "phase5_comparison_table_paper_ready.png")

# If you produced these earlier in the notebook, include them too
_maybe_copy(RUN_DIR / "phase5_model1_resolution_sensitivity.csv")
_maybe_copy(RUN_DIR / "phase5_comparison_table.csv")
_maybe_copy(FIG_DIR / "phase5_comparison_table.png")

zip_base = RUN_DIR / f"PHASE5_DELIVERABLE_{stamp}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(deliver_dir))

print("Deliverable folder:", deliver_dir)
print("ZIP:", zip_path)
print("ZIP size (MB):", round(Path(zip_path).stat().st_size / (1024**2), 2))

Deliverable folder: /kaggle/working/output/phase5/phase5_compare/deliverable_20260226_225412
ZIP: /kaggle/working/output/phase5/phase5_compare/PHASE5_DELIVERABLE_20260226_225412.zip
ZIP size (MB): 0.12
